In [125]:
from pathlib import Path

import polars as pl

project_path = Path.cwd() / "output" / "project"

In [126]:
cnpt_name = "calcium ionized".replace(" ", "_").replace("/", "-")
cnpt_pattern = "(50808//Free Calcium)"

In [127]:
cnpt_directory = project_path / "workspace" / "concept" / cnpt_name / "1.0.0"

cnpt_paths = [str(p) for p in cnpt_directory.iterdir() if p.is_file()]

cnpt =  pl.scan_parquet(cnpt_paths[0])
cnpt = cnpt.with_columns(pl.col("time").dt.replace_time_zone("UTC"))
cnpt = cnpt.collect()

In [128]:
cnpt

subject_id,time,code,numeric_value,text_value,dataset,table
i64,"datetime[μs, UTC]",str,f32,str,str,str
10000117,2174-10-14 14:52:00 UTC,"""calcium_ionized//mmol/L""",1.23,"""1.23""","""mimic-iv""","""labevents"""
10000117,2175-01-27 16:28:00 UTC,"""calcium_ionized//mmol/L""",1.2,"""1.20""","""mimic-iv""","""labevents"""
10001884,2131-01-11 06:37:00 UTC,"""calcium_ionized//mmol/L""",1.25,"""1.25""","""mimic-iv""","""labevents"""
10001884,2131-01-13 02:28:00 UTC,"""calcium_ionized//mmol/L""",1.17,"""1.17""","""mimic-iv""","""labevents"""
10002013,2160-05-18 09:19:00 UTC,"""calcium_ionized//mmol/L""",1.16,"""1.16""","""mimic-iv""","""labevents"""
…,…,…,…,…,…,…
19999840,2164-09-17 08:11:00 UTC,"""calcium_ionized//mmol/L""",1.15,"""1.15""","""mimic-iv""","""labevents"""
19999840,2164-09-17 13:18:00 UTC,"""calcium_ionized//mmol/L""",1.08,"""1.08""","""mimic-iv""","""labevents"""
19999840,2164-09-17 13:34:00 UTC,"""calcium_ionized//mmol/L""",1.15,"""1.15""","""mimic-iv""","""labevents"""


In [129]:
labevent = pl.scan_parquet(project_path / "workspace" / "extraction" / "mimic-iv" / "3.1" / "labevents" / "result.parquet")
data = labevent.filter(pl.col("code").str.contains(cnpt_pattern)).collect()

In [130]:
data.sort("time")

subject_id,time,code,numeric_value,text_value,hadm_id,stay_id
i64,datetime[μs],str,f32,str,str,str
16904137,2105-10-05 11:44:00,"""mimic-iv//3.1//labevents//resu…",1.04,"""1.04""","""21081215""","""-1"""
16904137,2105-10-05 12:41:00,"""mimic-iv//3.1//labevents//resu…",0.99,"""___""","""21081215""","""-1"""
11556680,2109-06-27 12:23:00,"""mimic-iv//3.1//labevents//resu…",1.16,"""1.16""",null,"""-1"""
12418411,2109-07-09 11:03:00,"""mimic-iv//3.1//labevents//resu…",1.01,"""1.01""",null,"""-1"""
12418411,2109-07-09 14:06:00,"""mimic-iv//3.1//labevents//resu…",1.15,"""1.15""",null,"""-1"""
…,…,…,…,…,…,…
15173387,2214-07-21 11:54:00,"""mimic-iv//3.1//labevents//resu…",1.2,"""1.20""","""28006722""","""-1"""
15173387,2214-07-23 19:28:00,"""mimic-iv//3.1//labevents//resu…",1.15,"""1.15""","""28006722""","""-1"""
12965924,2214-10-23 06:27:00,"""mimic-iv//3.1//labevents//resu…",1.12,"""1.12""",null,"""-1"""


In [131]:
for e in data.unique("code")["code"]:
    print(e)

mimic-iv//3.1//labevents//result//50808//Free Calcium//mmol/L


In [132]:
ricu = pl.read_parquet("./data/cnpt_labevents.parquet")
ricu

FileNotFoundError: No such file or directory (os error 2): ./data/cnpt_labevents.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'


In [ ]:
#ricu.sort("charttime_origin")

In [ ]:
ricu.join(cnpt, left_on=["subject_id", "charttime"], right_on=["subject_id", "time"], how="anti")


labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
i32,i32,i32,i32,i32,str,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,f64,f64,str,str,str
154,10000032,22595853,86271148,50971,null,2180-05-07 05:05:00 UTC,2180-05-07 07:03:00 UTC,"""4.5""",4.5,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
181,10000032,null,19543630,50971,"""P85UQ1""",2180-06-03 12:00:00 UTC,2180-06-03 13:04:00 UTC,"""3.3""",3.3,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
190,10000032,null,58691952,50971,"""P69FQC""",2180-06-03 12:00:00 UTC,2180-06-03 13:04:00 UTC,"""3.4""",3.4,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
403,10000032,29079034,55621508,50971,null,2180-07-23 21:45:00 UTC,2180-07-23 22:30:00 UTC,"""4.7""",4.7,"""mEq/L""",3.3,5.1,null,"""STAT""",null
451,10000032,29079034,66433308,50971,null,2180-07-25 04:45:00 UTC,2180-07-25 07:44:00 UTC,"""5.2""",5.2,"""mEq/L""",3.3,5.1,"""abnormal""","""ROUTINE""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
159057396,19999987,23865745,34834852,50971,null,2145-11-04 05:01:00 UTC,2145-11-04 05:51:00 UTC,"""___""",4.1,"""mEq/L""",3.3,5.1,null,"""ROUTINE""","""HEMOLYSIS FALSELY ELEVATES K.."""
159057461,19999987,23865745,62584889,50971,null,2145-11-05 06:10:00 UTC,2145-11-05 07:02:00 UTC,"""3.6""",3.6,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
159057486,19999987,23865745,45769962,50971,null,2145-11-06 10:07:00 UTC,2145-11-06 11:19:00 UTC,"""3.7""",3.7,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null


In [ ]:
cnpt.join(ricu, left_on=["subject_id", "time"], right_on=["subject_id", "charttime"], how="anti")

subject_id,time,code,numeric_value,text_value,dataset,table
i64,"datetime[μs, UTC]",str,f32,str,str,str
10000117,2174-09-22 12:26:00 UTC,"""albumin//g/dL""",4.7,"""4.7""","""mimic-iv""","""labevents"""
10000117,2175-01-27 16:15:00 UTC,"""albumin//g/dL""",4.7,"""4.7""","""mimic-iv""","""labevents"""
10000117,2176-02-08 22:00:00 UTC,"""albumin//g/dL""",4.9,"""4.9""","""mimic-iv""","""labevents"""
10000117,2176-09-08 10:00:00 UTC,"""albumin//g/dL""",4.9,"""4.9""","""mimic-iv""","""labevents"""
10000117,2178-06-12 15:19:00 UTC,"""albumin//g/dL""",4.8,"""4.8""","""mimic-iv""","""labevents"""
…,…,…,…,…,…,…
19999784,2119-06-27 05:40:00 UTC,"""albumin//g/dL""",3.7,"""3.7""","""mimic-iv""","""labevents"""
19999784,2119-07-07 17:30:00 UTC,"""albumin//g/dL""",4.3,"""4.3""","""mimic-iv""","""labevents"""
19999784,2123-04-12 12:28:00 UTC,"""albumin//g/dL""",4.0,"""4.0""","""mimic-iv""","""labevents"""


In [ ]:
# ricu.join(cnpt, left_on=["subject_id", "charttime", "valuenum"], right_on=["subject_id", "time", "numeric_value"], how="anti")

In [ ]:
# cnpt.join(ricu, left_on=["subject_id", "time", "numeric_value"], right_on=["subject_id", "charttime", "valuenum"], how="anti")